<a href="https://colab.research.google.com/github/shyjuwilsonskt/deepfake-forensics-lab/blob/main/02_spl_papr_benchmarks.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Check Colab Pro GPU assignment
!nvidia-smi

import os
import sys
import numpy as np
import matplotlib.pyplot as plt

# Ensure src modules are discoverable
if os.path.exists('/content/deepfake-forensics-lab'):
    sys.path.append('/content/deepfake-forensics-lab')
    os.chdir('/content/deepfake-forensics-lab')
elif os.path.exists('../src'):
    sys.path.append('..')

from src.residuals import extract_spatial_residual_np
from src.radial_profiling import compute_radial_profile_np
from src.metrics import compute_papr, evaluate_forensic_classifier
print("Modules successfully loaded.")

In [ ]:
import glob
import time
import cv2

def run_spl_evaluation(real_image_paths, fake_image_paths, r_min=10):
    real_paprs = []
    fake_paprs = []

    t0 = time.time()
    for path in real_image_paths:
        img = cv2.imread(path)
        if img is None:
            continue
        res = extract_spatial_residual_np(img, kernel_type="laplacian_8")
        profile = compute_radial_profile_np(res)
        real_paprs.append(compute_papr(profile, r_min=r_min))

    for path in fake_image_paths:
        img = cv2.imread(path)
        if img is None:
            continue
        res = extract_spatial_residual_np(img, kernel_type="laplacian_8")
        profile = compute_radial_profile_np(res)
        fake_paprs.append(compute_papr(profile, r_min=r_min))

    total_time = (time.time() - t0) / (len(real_paprs) + len(fake_paprs) + 1e-8) * 1000.0
    results = evaluate_forensic_classifier(real_paprs, fake_paprs)
    results["Latency_ms"] = float(total_time)
    return results, real_paprs, fake_paprs

print("Benchmark evaluation engine ready.")

In [ ]:
# IEEE Standard Plotting Setup
plt.rcParams.update({
    "font.size": 10,
    "axes.labelsize": 11,
    "legend.fontsize": 9,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "figure.figsize": (6.5, 3.2),
    "lines.linewidth": 1.5
})

os.makedirs("outputs", exist_ok=True)

# Example visualization generation
r_axis = np.arange(128)
# Substitute actual radial profile vectors here
fig, ax = plt.subplots(figsize=(6.5, 3.2), dpi=300)
ax.set_xlabel(r"Radial Frequency Bin Radius ($r$)")
ax.set_ylabel(r"$\log_{10} S_R(r)$ (Residual Power)")
ax.set_title("1D Azimuthal Radial Power Spectrum Comparison")
ax.grid(True, linestyle=":", alpha=0.6)

plt.tight_layout()
plt.savefig("outputs/fig_1d_radial_spectrum_ieee.pdf", format="pdf", dpi=300)
plt.savefig("outputs/fig_1d_radial_spectrum_ieee.png", format="png", dpi=300)
plt.show()
print("Plots saved in outputs/ directory.")